In [ ]:
# -----------------------------          LOAD IN RADAR           -------------

In [ ]:
# This is loading in swath data

In [ ]:
import glob

data_dir = "/nfs/tcdynasty/tang/tcradar/"

nc_files = sorted(glob.glob(f"{data_dir}/*.nc"))

print("Available files:")
for f in nc_files:
    print(f)


In [ ]:
import xarray as xr

base_dir = "/nfs/tcdynasty/tang/tcradar/"

merge_file = base_dir + "tc_radar_v3m_1997_2019_xy_rel_merge_ships.nc"
swath_file = base_dir + "tc_radar_v3m_1997_2019_xy_rel_swath_ships.nc"

ds_merge = xr.open_dataset(merge_file)
ds_swath = xr.open_dataset(swath_file)

print("MERGE dataset:")
print(ds_merge)

print("\nSWATH dataset:")
print(ds_swath)


In [ ]:
[var for var in ds_merge.data_vars if "storm" in var.lower()
 or "name" in var.lower()
 or "year" in var.lower()
 or "basin" in var.lower()]


In [ ]:
[var for var in ds_merge.coords]


In [ ]:
ds_merge["storm_name"].isel(num_cases=slice(0,10))


In [ ]:
import numpy as np

storm_names = np.char.upper(ds_merge["storm_name"].values.astype(str))
years = ds_merge["merge_year"].values

eduard_idx = np.where(
    (storm_names == "EDOUARD") & (years == 2014)
)[0]

eduard_idx


In [ ]:
ds_edouard = ds_merge.isel(num_cases=[114, 115, 116, 117, 118, 119])

print(ds_edouard)


In [ ]:
def print_edouard_case_info(ds):
    """
    Print summary info for each Edouard 2014 case,
    including num_cases, number_of_swaths, and any time-related attributes.
    """
    print("Summary of Edouard 2014 cases:\n")
    for i, idx in enumerate(ds["num_cases"].values):
        case = ds.isel(num_cases=i)
        print(f"Case {i} (num_cases={idx}):")
        # Number of swaths
        if "number_of_swaths" in case.data_vars:
            n_sw = case["number_of_swaths"].values
            print(f"  Number of swaths: {n_sw}")
        # Reflectivity range as a simple check
        if "total_recentered_reflectivity" in case.data_vars:
            refl = case["total_recentered_reflectivity"]
            print(f"  Reflectivity shape: {refl.shape}")
            print(f"  Max reflectivity: {np.nanmax(refl.values):.1f} dBZ")
            print(f"  Min reflectivity: {np.nanmin(refl.values):.1f} dBZ")
        # Check for any time-related attributes
        time_attrs = {k: v for k, v in case.attrs.items() if "time" in k.lower() or "date" in k.lower()}
        if time_attrs:
            print(f"  Time info (from attributes): {time_attrs}")
        else:
            print("  Time info: Not stored in dataset")
        print("-"*50)

# Use the function
print_edouard_case_info(ds_edouard)


In [ ]:
import numpy as np

print("Storm:", np.unique(ds_edouard["storm_name"].values))
print("Year:", np.unique(ds_edouard["merge_year"].values))
print("Number of cases:", ds_edouard.dims["num_cases"])


In [ ]:
[var for var in ds_edouard.data_vars
 if "time" in var.lower()
 or "date" in var.lower()
 or "hour" in var.lower()
 or "jul" in var.lower()]


In [ ]:
[var for var in ds_edouard.coords
 if "time" in var.lower()
 or "date" in var.lower()]


In [ ]:
ds_edouard["merge_hour"].values


In [ ]:
import numpy as np

merge_hours = ds_edouard["merge_hour"].values

# Find closest to 5Z
idx_05z = np.argmin(np.abs(merge_hours - 5))
idx_05z


In [ ]:
case_05z = ds_edouard.isel(num_cases=idx_05z)
print(case_05z)


In [ ]:
# -----------------------              LOADED IN, NOW WE MAKE PLOTS       ------

In [ ]:
import numpy as np

# recentered coordinates (storm center at 0,0)
x = case_05z["eastward_distance"].values   # km
y = case_05z["northward_distance"].values  # km

# 2D radius grid
X, Y = np.meshgrid(x, y)
R = np.sqrt(X**2 + Y**2)  # distance from storm center


In [ ]:
fr_mask = (X > 0) & (Y > 0)


In [ ]:
radar_var = case_05z["total_recentered_reflectivity"].values

# Expand mask along height axis
fr_mask_3d = np.repeat(
    fr_mask[:, :, np.newaxis],
    radar_var.shape[2],
    axis=2
)

# Apply mask
radar_var_fr = np.where(fr_mask_3d, radar_var, np.nan)


In [ ]:
RMW = 27.78  # km

r_min = 2.0 * RMW     # ~55 km
r_max = 6.0 * RMW     # ~165 km

outer_mask = (R >= r_min) & (R <= r_max)


In [ ]:
print([name for name in globals() if 'fr' in name])


In [ ]:
outer_fr_mask = fr_mask & outer_mask

outer_fr_mask_3d = np.repeat(
    outer_fr_mask[:, :, np.newaxis],
    radar_var_fr.shape[2],
    axis=2
)

radar_var_outer_fr = np.where(
    outer_fr_mask_3d,
    radar_var_fr,
    np.nan
)


In [ ]:
#computing radial bins
r_bins = np.linspace(r_min, r_max, 50)
r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])


In [ ]:
nz = radar_var_outer_fr.shape[2]
radial_profile = np.full((len(r_centers), nz), np.nan)

for i in range(len(r_centers)):
    bin_mask = (R >= r_bins[i]) & (R < r_bins[i+1])

    # broadcasting instead of repeat
    masked = np.where(
        bin_mask[:, :, np.newaxis],
        radar_var_outer_fr,
        np.nan
    )

    for k in range(nz):
        layer = masked[:, :, k]
        if np.any(~np.isnan(layer)):
            radial_profile[i, k] = np.nanmean(layer)


In [ ]:
print("Any non-NaN:", np.any(~np.isnan(radial_profile)))
print("Heights with data:", np.sum(~np.isnan(radial_profile), axis=0))


In [ ]:
#trim the heights
valid_heights = np.where(np.sum(~np.isnan(radial_profile), axis=0) > 0)[0]

zmin = valid_heights[0]
zmax = valid_heights[-1] + 1

radial_profile_trim = radial_profile[:, zmin:zmax]
height_trim = case_05z["height"].values[zmin:zmax]


In [ ]:
#radius–height cross section azimuthally averaged over a selected part of the storm
#It shows how your chosen radar variable changes with height as you move outward from the storm center
#, averaged over the front-right outer rainband region



import matplotlib.pyplot as plt

plt.figure(figsize=(7,8))

plt.pcolormesh(
    r_centers,
    height_trim,
    radial_profile_trim.T,
    shading="auto"
)

plt.axvline(27.78, color="k", linestyle="--", label="RMW")

plt.xlabel("Radius (km)")
plt.ylabel("Height (km)")
plt.title("EDOUARD (2014) – Front-Right Outer Rainband")
plt.colorbar(label="Mean Reflectivity (dBZ)")

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# -----------------           Understanding how the cases are arranged  -------

In [ ]:
print("num_cases dimension:", case_05z["num_cases"].values)


In [ ]:
import numpy as np

def list_edouard_cases(ds_merge):
    """
    Print a summary of all Edouard 2014 cases in the merged dataset.
    Shows num_cases, number of swaths, reflectivity range.
    """
    # Extract all Edouard 2014 indices
    storm_names = np.char.upper(ds_merge["storm_name"].values.astype(str))
    years = ds_merge["merge_year"].values
    eduard_idx = np.where((storm_names == "EDOUARD") & (years == 2014))[0]

    print(f"Found {len(eduard_idx)} Edouard 2014 cases:\n")
    
    for i, idx in enumerate(eduard_idx):
        case = ds_merge.isel(num_cases=idx)
        num_swaths = case["number_of_swaths"].values if "number_of_swaths" in case.data_vars else "N/A"
        refl = case["total_recentered_reflectivity"].values
        print(f"Case {i} (num_cases={idx}):")
        print(f"  Number of swaths: {num_swaths}")
        print(f"  Reflectivity shape: {refl.shape}")
        print(f"  Max reflectivity: {np.nanmax(refl):.1f} dBZ")
        print(f"  Min reflectivity: {np.nanmin(refl):.1f} dBZ")
        print("-"*50)

# Usage
list_edouard_cases(ds_merge)


In [ ]:
print(case_05z)


In [ ]:
# -----------------           Just plotting the radar           ------------

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Select radar variable
radar_var = case_05z["total_recentered_reflectivity"].values

# 2. Grid (storm-relative, km)
x = case_05z["eastward_distance"].values
y = case_05z["northward_distance"].values
X, Y = np.meshgrid(x, y)

# 3. Height selection
height_km = case_05z["height"].values
z_level = 2.0   # km (start low!)
iz = np.argmin(np.abs(height_km - z_level))

print(f"Plotting z = {height_km[iz]:.2f} km")

# 4. Plot
plt.figure(figsize=(6,6))
plt.pcolormesh(
    X, Y,
    radar_var[:, :, iz],
    shading="auto",
    cmap="turbo",
    vmin=0, vmax=50   # reflectivity sanity range
)
plt.colorbar(label="Reflectivity (dBZ)")
plt.scatter(0, 0, c="k", marker="+", s=80)  # TC center
plt.axis("equal")
plt.xlabel("Eastward distance (km)")
plt.ylabel("Northward distance (km)")
plt.title("Edouard (2014) – Radar Reflectivity\nCase 119")
plt.show()


In [ ]:
# -----------------------------    creating annimations    -------------------

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Radar variable and coordinates
radar_var = case_05z["total_recentered_reflectivity"].values
x = case_05z["eastward_distance"].values
y = case_05z["northward_distance"].values
X, Y = np.meshgrid(x, y)

height_km = case_05z["height"].values
z_levels = [0.5, 1.0, 2.0, 3.0, 4.0]  # example heights in km
iz_list = [np.argmin(np.abs(height_km - z)) for z in z_levels]

fig, ax = plt.subplots(figsize=(6,6))
cmap = plt.get_cmap("turbo")
mesh = ax.pcolormesh(X, Y, radar_var[:, :, iz_list[0]], shading="auto", cmap=cmap, vmin=0, vmax=50)
cb = plt.colorbar(mesh, ax=ax, label="Reflectivity (dBZ)")
ax.scatter(0, 0, c="k", marker="+", s=80)  # TC center
ax.axis("equal")
ax.set_xlabel("Eastward distance (km)")
ax.set_ylabel("Northward distance (km)")
title = ax.set_title(f"Edouard (2014) – Radar Reflectivity\nz = {height_km[iz_list[0]]:.2f} km")

def update(frame):
    iz = iz_list[frame]
    mesh.set_array(radar_var[:, :, iz].ravel())
    title.set_text(f"Edouard (2014) – Radar Reflectivity\nz = {height_km[iz]:.2f} km")
    return mesh, title

ani = FuncAnimation(fig, update, frames=len(iz_list), interval=500, blit=False)
plt.show()


In [ ]:
#                THIS IS VERTICAL DIFFERENCE AT ONE TIME             



import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Radar variable and coordinates
radar_var = case_05z["total_recentered_reflectivity"].values
x = case_05z["eastward_distance"].values
y = case_05z["northward_distance"].values
X, Y = np.meshgrid(x, y)

height_km = case_05z["height"].values
z_levels = [0.5, 1.0, 2.0, 3.0, 4.0]  # example heights in km
iz_list = [np.argmin(np.abs(height_km - z)) for z in z_levels]

fig, ax = plt.subplots(figsize=(6,6))
cmap = plt.get_cmap("turbo")

# Initial frame
mesh = ax.pcolormesh(X, Y, radar_var[:, :, iz_list[0]], shading="auto", cmap=cmap, vmin=0, vmax=50)
cb = plt.colorbar(mesh, ax=ax, label="Reflectivity (dBZ)")
ax.scatter(0, 0, c="k", marker="+", s=80)  # TC center
ax.axis("equal")
ax.set_xlabel("Eastward distance (km)")
ax.set_ylabel("Northward distance (km)")
title = ax.set_title(f"Edouard (2014) – Radar Reflectivity\nz = {height_km[iz_list[0]]:.2f} km")

def update(frame):
    iz = iz_list[frame]
    # Update the mesh array
    mesh.set_array(radar_var[:, :, iz].ravel())
    title.set_text(f"Edouard (2014) – Radar Reflectivity\nz = {height_km[iz]:.2f} km")
    return mesh, title

# Create animation
ani = FuncAnimation(fig, update, frames=len(iz_list), interval=500, blit=False)

# Save as MP4
ani.save("radar_loop.mp4", writer="ffmpeg", fps=2)  # fps = frames per second

plt.close(fig)  # close the figure to avoid showing static plot


In [ ]:
# Plane flies along the x-axis at y = 0 (middle)
y_center_idx = len(y)//2
swath_width_km = 150  # width perpendicular to flight, adjust as needed
dy = y[1] - y[0]
swath_width_pts = int(swath_width_km / dy)

# Indices for swath perpendicular to flight, clipped to array bounds
y_start = max(0, y_center_idx - swath_width_pts//2)
y_end = min(len(y), y_center_idx + swath_width_pts//2)


In [ ]:
#-------------------------- New work, 02/12   ------------------------

In [ ]:
#Making sure we are getting the timing convention correct 

In [ ]:
print("Merge ships_lag_times:", ds_merge['ships_lag_times'].values)
print("Swath ships_lag_times:", ds_swath['ships_lag_times'].values)


In [ ]:
import pandas as pd
import numpy as np

# pick a case index
case_idx = 0

# get the case midpoint time from merge_year, merge_month, etc.
midpoint = pd.Timestamp(
    year=int(ds_merge['merge_year'][case_idx].values),
    month=int(ds_merge['merge_month'][case_idx].values),
    day=int(ds_merge['merge_day'][case_idx].values),
    hour=int(ds_merge['merge_hour'][case_idx].values),
    minute=int(ds_merge['merge_minute'][case_idx].values)
)

# get ships lag times (hours)
lag_hours = ds_merge['ships_lag_times'].values

# compute actual timestamps for each lag
actual_times = midpoint + pd.to_timedelta(lag_hours, unit='h')
print("Actual UTC times for case 0:")
print(actual_times)


In [ ]:
# Make sure storm_name is uppercase to match dataset convention
target_storm = "EDOUARD"
target_year = 2014

# Find case indices matching this storm and year
case_indices = np.where(
    (ds_merge['storm_name'].values.astype(str) == target_storm) &
    (ds_merge['merge_year'].values == target_year)
)[0]

print("Case indices for Hurricane Edouard 2014:", case_indices)


In [ ]:
# 6 separate analyses for Hurricane Edouard 2014 -------------------

In [ ]:
import pandas as pd

case_idx = 119  # pick the last Edouard case

# midpoint time
midpoint = pd.Timestamp(
    year=int(ds_merge['merge_year'][case_idx].values),
    month=int(ds_merge['merge_month'][case_idx].values),
    day=int(ds_merge['merge_day'][case_idx].values),
    hour=int(ds_merge['merge_hour'][case_idx].values),
    minute=int(ds_merge['merge_minute'][case_idx].values)
)

# ships lag times
lag_hours = ds_merge['ships_lag_times'].values

# actual UTC times
actual_times = midpoint + pd.to_timedelta(lag_hours, unit='h')
print(f"UTC times for Hurricane Edouard 2014, case {case_idx}:")
print(actual_times)


In [ ]:
# -------------       Plotting, THIS IS SHIPS DATA WITHIN MERGED     ---------------

In [ ]:
reflectivity = ds_merge['recentered_reflectivity'][case_idx].values  # shape: (northward_distance, eastward_distance, height)
wind_speed = ds_merge['recentered_wind_speed'][case_idx].values


In [ ]:
for i, t in enumerate(actual_times):
    print(f"Lag {ds_merge['ships_lag_times'][i].values} h → UTC {t}")
    # access any variable at this lag, e.g., ohc_ships:
    ohc = ds_merge['ohc_ships'][case_idx, i].values
    print("OHC:", ohc)


In [ ]:
import matplotlib.pyplot as plt

lag_hours = ds_merge['ships_lag_times'].values
ohc_values = ds_merge['ohc_ships'][case_idx].values  # case 119

plt.plot(actual_times, ohc_values, marker='o')
plt.xlabel("UTC Time")
plt.ylabel("OHC (J kg⁻¹ °C⁻¹)")
plt.title("Hurricane Edouard 2014 - Case 119")
plt.grid(True)
plt.xticks(rotation=30)
plt.show()


In [ ]:
# -------------------     Now lets work with swath data         -----------

In [ ]:
print(ds_swath['ships_lag_times'].values)


In [ ]:
edouard_cases = [114, 115, 116, 117, 118, 119]
case_idx = 119  # pick the last case for example


In [ ]:
lags = ds_swath['ships_lag_times'].values  # e.g., [-48, -42, ... , 48]

# Extract the best-track times for this case
case = case_idx
utc_times = pd.to_datetime({
    'year': ds_swath['year_ships'].values[case, :],
    'month': ds_swath['month_ships'].values[case, :],
    'day': ds_swath['day_ships'].values[case, :],
    'hour': ds_swath['hour_ships'].values[case, :],
})

# Print results
for lag, utc in zip(lags, utc_times):
    print(f"Lag {lag:.1f} h → UTC {utc}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

case = 119
lag_idx = 8  # Lag 0 h
height_idx = 0  # lowest level

# Extract the 2D reflectivity grid
reflectivity_grid = ds_swath['recentered_reflectivity'].values[case, :, :, height_idx]

# Mask NaNs
reflectivity_grid = np.ma.masked_invalid(reflectivity_grid)

# Get coordinates
north = ds_swath['northward_distance'].values
east = ds_swath['eastward_distance'].values

# Plot
plt.figure(figsize=(7,6))
c = plt.pcolormesh(east, north, reflectivity_grid, shading='auto', cmap='turbo', vmin=0, vmax=60)
plt.colorbar(c, label='Reflectivity (dBZ)')
plt.xlabel('Eastward Distance (km)')
plt.ylabel('Northward Distance (km)')
plt.title(f'Hurricane Edouard 2014 - Case {case} - Lag 0 h')
plt.show()


In [ ]:
case = 119
height_idx = 0  # lowest level

# Extract the reflectivity grid for all lags at this height
reflectivity_all_lags = ds_swath['recentered_reflectivity'].values[case, :, :, height_idx]

# Check type and shape
print("Type of reflectivity:", type(reflectivity_all_lags))
print("Shape of reflectivity grid:", reflectivity_all_lags.shape)

# Check min, max, and number of NaNs
print("Min value:", np.nanmin(reflectivity_all_lags))
print("Max value:", np.nanmax(reflectivity_all_lags))
print("Number of NaNs:", np.sum(np.isnan(reflectivity_all_lags)))

# Check coordinates
print("Northward distance shape:", ds_swath['northward_distance'].values.shape)
print("Eastward distance shape:", ds_swath['eastward_distance'].values.shape)
print("North range:", ds_swath['northward_distance'].values[:5], "...")
print("East range:", ds_swath['eastward_distance'].values[:5], "...")


In [ ]:
for h in range(len(ds_swath['height'])):
    refl = ds_swath['recentered_reflectivity'].values[case, :, :, h]
    n_nans = np.sum(np.isnan(refl))
    print(f"Height {ds_swath['height'].values[h]} km → NaNs: {n_nans} / {refl.size}")


In [ ]:
import matplotlib.pyplot as plt

case = 119
height_idx = np.argmin(np.abs(ds_swath['height'].values - 1.0))  # ~1 km
lags = ds_swath['ships_lag_times'].values  # e.g., [-48, -42, ..., 48]

# Inspect shape
refl_data = ds_swath['recentered_reflectivity'].values[case]  # pick case first
print("Shape of reflectivity array for case:", refl_data.shape)


In [ ]:
import numpy as np

# Pick the year and month arrays
years = ds_swath['year_ships'].values  # shape (num_cases, num_lags)
months = ds_swath['month_ships'].values

# Find swath indices where any lag has year=2014 and month=9
edouard_swath_indices = []
for i in range(years.shape[0]):
    if np.any((years[i, :] == 2014) & (months[i, :] == 9)):
        edouard_swath_indices.append(i)

print("Swath indices corresponding to Edouard 2014:", edouard_swath_indices)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

#Swath indices corresponding to Edouard 2014: [350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362]

# --- Settings ---
case = 351             # Hurricane Edouard 2014
height_km = 1.0        # height to plot
height_idx = np.argmin(np.abs(ds_swath['height'].values - height_km))  # find closest height
lags = ds_swath['ships_lag_times'].values
utc_times = pd.to_datetime({
    'year': ds_swath['year_ships'].values[case, :],
    'month': ds_swath['month_ships'].values[case, :],
    'day': ds_swath['day_ships'].values[case, :],
    'hour': ds_swath['hour_ships'].values[case, :],
})

# --- Loop through all lags ---
for lag_idx, lag in enumerate(lags):
    # Extract 2D reflectivity grid for this lag
    refl_grid = ds_swath['recentered_reflectivity'].values[case, :, :, height_idx]
    
    # If you want, you can select a specific lag with lag_idx:
    # refl_grid = ds_swath['recentered_reflectivity'].values[case, :, :, height_idx, lag_idx]

    # Plot
    plt.figure(figsize=(6,5))
    im = plt.imshow(refl_grid, origin='lower', 
                    extent=[ds_swath['eastward_distance'].values[0],
                            ds_swath['eastward_distance'].values[-1],
                            ds_swath['northward_distance'].values[0],
                            ds_swath['northward_distance'].values[-1]],
                    cmap='turbo', vmin=0, vmax=60)
    plt.colorbar(im, label='Reflectivity (dBZ)')
    plt.title(f'Case {case} | Height {height_km} km | Lag {lag} h | UTC {utc_times[lag_idx]}')
    plt.xlabel('Eastward Distance (km)')
    plt.ylabel('Northward Distance (km)')
    plt.show()


In [ ]:
# I assume Cartesian coordinates? 
print(ds_swath['recentered_reflectivity'].dims)
print(ds_swath['recentered_reflectivity'].shape)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Swath indices corresponding to Edouard 2014:
# [350, 351, 352, ..., 362]

case = 351
height_km = 1.0
height_idx = np.argmin(np.abs(ds_swath['height'].values - height_km))

# Build UTC for THIS swath only (single time)
utc_time = pd.to_datetime({
    'year': ds_swath['year_ships'].values[case],
    'month': ds_swath['month_ships'].values[case],
    'day': ds_swath['day_ships'].values[case],
    'hour': ds_swath['hour_ships'].values[case],
})

# Extract 2D reflectivity (no lag indexing)
refl_grid = ds_swath['recentered_reflectivity'].values[case, :, :, height_idx]

# Plot
plt.figure(figsize=(6,5))
im = plt.imshow(
    refl_grid,
    origin='lower',
    extent=[
        ds_swath['eastward_distance'].values[0],
        ds_swath['eastward_distance'].values[-1],
        ds_swath['northward_distance'].values[0],
        ds_swath['northward_distance'].values[-1]
    ],
    cmap='turbo',
    vmin=0, vmax=60
)

plt.colorbar(im, label='Reflectivity (dBZ)')
plt.title(f'Edouard 2014 | Swath {case} | Height {height_km} km | UTC {utc_time}')
plt.xlabel('Eastward Distance (km)')
plt.ylabel('Northward Distance (km)')
plt.show()


In [ ]:
edouard_swaths = [350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362]

for case in edouard_swaths:

    utc_time = pd.Timestamp(
        year=int(ds_swath['year_ships'].values[case, 0]),
        month=int(ds_swath['month_ships'].values[case, 0]),
        day=int(ds_swath['day_ships'].values[case, 0]),
        hour=int(ds_swath['hour_ships'].values[case, 0])
    )

    refl_grid = ds_swath['recentered_reflectivity'].values[case, :, :, height_idx]

    plt.figure(figsize=(6,5))
    im = plt.imshow(
        refl_grid,
        origin='lower',
        extent=[
            ds_swath['eastward_distance'].values[0],
            ds_swath['eastward_distance'].values[-1],
            ds_swath['northward_distance'].values[0],
            ds_swath['northward_distance'].values[-1]
        ],
        cmap='turbo', vmin=0, vmax=60
    )

    plt.colorbar(im, label='Reflectivity (dBZ)')
    plt.title(f'Edouard 2014 | Swath {case} | 1 km | UTC {utc_time}')
    plt.xlabel('Eastward Distance (km)')
    plt.ylabel('Northward Distance (km)')
    plt.show()


In [ ]:
#----------     Treat like Cartesian grid  -----------------

In [ ]:
print(ds_swath['recentered_reflectivity'].dims)
print(ds_swath['recentered_reflectivity'].shape)


In [ ]:
#number of cases
#eastward_distance → x-axis (km from storm center)
#northward_distance → y-axis (km from storm center)
#height → vertical (km)


In [ ]:
case = 351

# Coordinates
east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# Find center index (northward = 0 km)
center_idx = np.argmin(np.abs(north - 0))

# Heights between 1 and 2 km
z_mask = (height >= 1.0) & (height <= 2.0)
z_indices = np.where(z_mask)[0]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

case = 351

# Coordinates
east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# Find center of storm (north = 0 km)
center_north_idx = np.argmin(np.abs(north - 0))

# Extract full vertical column cross section
cross_section = ds_swath['recentered_reflectivity'].values[
    case,
    center_north_idx,
    :,   # all eastward distances
    :    # all heights
].T  # transpose so shape = (height, east)

# Extract 2D plan view at 1 km for context
height_idx = np.argmin(np.abs(height - 1.0))

plan_view = ds_swath['recentered_reflectivity'].values[
    case,
    :,
    :,
    height_idx
]

# ---- Plot Side-by-Side ----
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# LEFT: Plan View
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)
axes[0].axhline(0, color='black', linewidth=2)
axes[0].set_title('Plan View (1 km)')
axes[0].set_xlabel('Eastward Distance (km)')
axes[0].set_ylabel('Northward Distance (km)')
fig.colorbar(im1, ax=axes[0], label='dBZ')

# RIGHT: Vertical Cross Section
im2 = axes[1].pcolormesh(
    east,
    height,
    cross_section,
    shading='auto',
    cmap='turbo',
    vmin=0, vmax=60
)
axes[1].set_title('Vertical Cross Section (North = 0 km)')
axes[1].set_xlabel('Eastward Distance (km)')
axes[1].set_ylabel('Height (km)')
fig.colorbar(im2, ax=axes[1], label='dBZ')

plt.tight_layout()
plt.show()


In [ ]:
# -------------------   Try to find timing that works with lighting flashes ----

In [ ]:
import pandas as pd
import numpy as np

target_time = pd.Timestamp(2014, 9, 14, 20)
time_tolerance = pd.Timedelta(minutes=30)

matching_cases = []
time_diffs = []

for i in range(ds_swath.sizes['num_cases']):
    year = ds_swath['year_ships'].values[i, 0]
    month = ds_swath['month_ships'].values[i, 0]
    day = ds_swath['day_ships'].values[i, 0]
    hour = ds_swath['hour_ships'].values[i, 0]

    if np.isnan(year) or np.isnan(month) or np.isnan(day) or np.isnan(hour):
        continue

    utc_time = pd.Timestamp(
        year=int(year),
        month=int(month),
        day=int(day),
        hour=int(hour)
    )

    time_diff = abs(utc_time - target_time)
    time_diffs.append((time_diff, i))

    if time_diff <= time_tolerance:
        matching_cases.append(i)

# Print matching cases
print("Matching case indices within ±30 min:", matching_cases)

# Find the nearest swath
if time_diffs:
    nearest_case = min(time_diffs, key=lambda x: x[0])[1]
    nearest_time = pd.Timestamp(
        year=int(ds_swath['year_ships'].values[nearest_case, 0]),
        month=int(ds_swath['month_ships'].values[nearest_case, 0]),
        day=int(ds_swath['day_ships'].values[nearest_case, 0]),
        hour=int(ds_swath['hour_ships'].values[nearest_case, 0])
    )
    print(f"Nearest swath index: {nearest_case}, UTC: {nearest_time}")


In [ ]:
# CHANGE THE ORIENTATION OF THE CROSS SECTION

In [ ]:
#---------   Lets check to see if shear is here    -------------

In [ ]:
for var in ds_swath.data_vars:
    if "ships" in var.lower() or "shear" in var.lower() or "shr" in var.lower():
        print(var)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator

case = 358

# -------------------------------------------------
# Get UTC time for this case
# -------------------------------------------------
year = int(ds_swath['year_ships'].values[case, 0])
month = int(ds_swath['month_ships'].values[case, 0])
day = int(ds_swath['day_ships'].values[case, 0])
hour = int(ds_swath['hour_ships'].values[case, 0])

utc_string = f"{year:04d}-{month:02d}-{day:02d} {hour:02d}Z"

# -------------------------------------------------
# Coordinates
# -------------------------------------------------
east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# -------------------------------------------------
# 1) Get shear magnitude + direction
# -------------------------------------------------
shear_mag = ds_swath['shdc_ships'].values[case, 0]     # kt
shear_dir_deg = ds_swath['sddc_ships'].values[case, 0] # degrees (meteorological)

# Convert meteorological direction (FROM) to math angle (TOWARD)
theta = np.deg2rad(270 - shear_dir_deg)

ux = np.cos(theta)
uy = np.sin(theta)

# -------------------------------------------------
# 2) Plan view at 1 km
# -------------------------------------------------
height_idx = np.argmin(np.abs(height - 1.0))

plan_view = ds_swath['recentered_reflectivity'].values[
    case, :, :, height_idx
]

# -------------------------------------------------
# 3) Build shear-oriented cross section
# -------------------------------------------------
max_radius = 200  # km
n_points = 400

s = np.linspace(-max_radius, max_radius, n_points)

x_line = s * ux
y_line = s * uy

interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_section = np.zeros((len(height), n_points))

for k in range(len(height)):
    slice2d = ds_swath['recentered_reflectivity'].values[
        case, :, :, k
    ]
    
    interp.values = slice2d
    points = np.column_stack((y_line, x_line))
    cross_section[k, :] = interp(points)

# -------------------------------------------------
# 4) Plot
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# ---- LEFT: Plan View ----
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

# Shear cross-section line
axes[0].plot(x_line, y_line, color='black', linewidth=2)

# Shear arrow
arrow_scale = 50
axes[0].arrow(
    0, 0,
    ux * arrow_scale,
    uy * arrow_scale,
    color='white',
    width=3
)

axes[0].set_title(f'Plan View (1 km)\nShear = {shear_mag:.1f} kt')
axes[0].set_xlabel('Eastward Distance (km)')
axes[0].set_ylabel('Northward Distance (km)')
fig.colorbar(im1, ax=axes[0], label='dBZ')

# ---- RIGHT: Shear-Aligned Cross Section ----
im2 = axes[1].pcolormesh(
    s,
    height,
    cross_section,
    shading='auto',
    cmap='turbo',
    vmin=0, vmax=60
)

axes[1].set_title('Vertical Cross Section (Along Shear)')
axes[1].set_xlabel('Distance Along Shear (km)')
axes[1].set_ylabel('Height (km)')
fig.colorbar(im2, ax=axes[1], label='dBZ')

# ---- Overall Title with Time ----
fig.suptitle(
    f"Shear-Oriented Cross Section | {utc_string} UTC",
    fontsize=14
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator

case = 358

# -------------------------------------------------
# Get UTC time for this case
# -------------------------------------------------
year = int(ds_swath['year_ships'].values[case, 0])
month = int(ds_swath['month_ships'].values[case, 0])
day = int(ds_swath['day_ships'].values[case, 0])
hour = int(ds_swath['hour_ships'].values[case, 0])

utc_string = f"{year:04d}-{month:02d}-{day:02d} {hour:02d}Z"

# -------------------------------------------------
# Coordinates
# -------------------------------------------------
east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# -------------------------------------------------
# 1) Get shear magnitude + direction
# -------------------------------------------------
shear_mag = ds_swath['shdc_ships'].values[case, 0]      # kt
shear_dir_deg = ds_swath['sddc_ships'].values[case, 0]  # degrees (meteorological, FROM)

# Convert meteorological direction (FROM) to math angle (TOWARD)
theta = np.deg2rad(270 - shear_dir_deg)

ux = np.cos(theta)
uy = np.sin(theta)

# -------------------------------------------------
# 2) Plan view at 1 km
# -------------------------------------------------
height_idx = np.argmin(np.abs(height - 1.0))

plan_view = ds_swath['recentered_reflectivity'].values[
    case, :, :, height_idx
]

# -------------------------------------------------
# 3) Build shear-oriented cross section
# -------------------------------------------------
max_radius = 200  # km
n_points = 400

s = np.linspace(-max_radius, max_radius, n_points)

x_line = s * ux
y_line = s * uy

interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_section = np.zeros((len(height), n_points))

for k in range(len(height)):
    slice2d = ds_swath['recentered_reflectivity'].values[
        case, :, :, k
    ]
    
    interp.values = slice2d
    points = np.column_stack((y_line, x_line))
    cross_section[k, :] = interp(points)

# -------------------------------------------------
# 4) Plot
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# ---- LEFT: Plan View ----
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

axes[0].plot(x_line, y_line, color='black', linewidth=2)

arrow_scale = 50
axes[0].arrow(
    0, 0,
    ux * arrow_scale,
    uy * arrow_scale,
    color='white',
    width=3
)

axes[0].set_title(f'Plan View (1 km)')
axes[0].set_xlabel('Eastward Distance (km)')
axes[0].set_ylabel('Northward Distance (km)')
fig.colorbar(im1, ax=axes[0], label='dBZ')

# ---- RIGHT: Shear-Aligned Cross Section ----
im2 = axes[1].pcolormesh(
    s,
    height,
    cross_section,
    shading='auto',
    cmap='turbo',
    vmin=0, vmax=60
)

axes[1].set_title('Vertical Cross Section (Along Shear)')
axes[1].set_xlabel('Distance Along Shear (km)')
axes[1].set_ylabel('Height (km)')
fig.colorbar(im2, ax=axes[1], label='dBZ')

# ---- Overall Title with Time ----
fig.suptitle(
    f"Shear-Oriented Cross Section | {utc_string} UTC",
    fontsize=14
)

# ---- Shear Info Under Plot ----
fig.text(
    0.5, -0.02,
    f"Deep-layer Shear = {shear_mag:.1f} kt | Direction (FROM) = {shear_dir_deg:.0f}°",
    ha='center',
    fontsize=11
)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
# ADDING WWLLN DATA

In [ ]:
# first adding a grid to make sure lat and long looks good

In [ ]:
storm_lat = ds_swath['lat_ships'].values[case, 0]
storm_lon = ds_swath['lon_ships'].values[case, 0]

In [ ]:
# Convert longitude if dataset uses 0–360 convention
# Convert 0–360 longitude to -180–180
storm_lon = -abs(storm_lon)

print(f"Storm center location:")
print(f"Latitude : {storm_lat:.2f}°")
print(f"Longitude: {storm_lon:.2f}°")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator

case = 358

# -------------------------------------------------
# Get UTC time for this case
# -------------------------------------------------
year = int(ds_swath['year_ships'].values[case, 0])
month = int(ds_swath['month_ships'].values[case, 0])
day = int(ds_swath['day_ships'].values[case, 0])
hour = int(ds_swath['hour_ships'].values[case, 0])

utc_string = f"{year:04d}-{month:02d}-{day:02d} {hour:02d}Z"

# -------------------------------------------------
# Coordinates
# -------------------------------------------------
east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# -------------------------------------------------
# 1) Get shear magnitude + direction
# -------------------------------------------------
shear_mag = ds_swath['shdc_ships'].values[case, 0]
shear_dir_deg = ds_swath['sddc_ships'].values[case, 0]

theta = np.deg2rad(270 - shear_dir_deg)
ux = np.cos(theta)
uy = np.sin(theta)

# -------------------------------------------------
# 2) Plan view at 1 km
# -------------------------------------------------
height_idx = np.argmin(np.abs(height - 1.0))

plan_view = ds_swath['recentered_reflectivity'].values[
    case, :, :, height_idx
]

# -------------------------------------------------
# 3) Build shear-oriented cross section
# -------------------------------------------------
max_radius = 200  # km
n_points = 400

s = np.linspace(-max_radius, max_radius, n_points)

x_line = s * ux
y_line = s * uy

interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_section = np.zeros((len(height), n_points))

for k in range(len(height)):
    slice2d = ds_swath['recentered_reflectivity'].values[
        case, :, :, k
    ]
    
    interp.values = slice2d
    points = np.column_stack((y_line, x_line))
    cross_section[k, :] = interp(points)

# -------------------------------------------------
# 4) Plot
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# ---- LEFT: Plan View ----
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

# Shear line + arrow
axes[0].plot(x_line, y_line, color='black', linewidth=2)

arrow_scale = 50
axes[0].arrow(
    0, 0,
    ux * arrow_scale,
    uy * arrow_scale,
    color='white',
    width=3
)

axes[0].set_title('Plan View (1 km)')
axes[0].set_xlabel('Eastward Distance (km)')
axes[0].set_ylabel('Northward Distance (km)')
fig.colorbar(im1, ax=axes[0], label='dBZ')

# ---- Add Plan View Grid ----
xmin, xmax = east[0], east[-1]
ymin, ymax = north[0], north[-1]

axes[0].set_xticks(np.arange(np.floor(xmin/50)*50, xmax+50, 50))
axes[0].set_yticks(np.arange(np.floor(ymin/50)*50, ymax+50, 50))

axes[0].set_xticks(np.arange(np.floor(xmin/25)*25, xmax+25, 25), minor=True)
axes[0].set_yticks(np.arange(np.floor(ymin/25)*25, ymax+25, 25), minor=True)

axes[0].grid(which='major', color='white', alpha=0.35, linestyle='--', linewidth=0.8)
axes[0].grid(which='minor', color='white', alpha=0.2, linestyle='--', linewidth=0.5)

# ---- RIGHT: Shear-Aligned Cross Section ----
im2 = axes[1].pcolormesh(
    s,
    height,
    cross_section,
    shading='auto',
    cmap='turbo',
    vmin=0, vmax=60
)

axes[1].set_title('Vertical Cross Section (Along Shear)')
axes[1].set_xlabel('Distance Along Shear (km)')
axes[1].set_ylabel('Height (km)')
fig.colorbar(im2, ax=axes[1], label='dBZ')

# ---- Add Cross Section Grid ----
axes[1].set_xticks(np.arange(-max_radius, max_radius+1, 50))
axes[1].set_yticks(np.arange(0, int(height.max())+1, 2))

axes[1].grid(color='white', alpha=0.3, linestyle='--', linewidth=0.8)

# ---- Overall Title ----
fig.suptitle(
    f"Shear-Oriented Cross Section | {utc_string} UTC",
    fontsize=14
)

# ---- Shear Info Under Plot ----
fig.text(
    0.5, -0.02,
    f"Deep-layer Shear = {shear_mag:.1f} kt | Direction (FROM) = {shear_dir_deg:.0f}°",
    ha='center',
    fontsize=11
)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator

case = 358

# -------------------------------------------------
# Get UTC time for this case
# -------------------------------------------------
year = int(ds_swath['year_ships'].values[case, 0])
month = int(ds_swath['month_ships'].values[case, 0])
day = int(ds_swath['day_ships'].values[case, 0])
hour = int(ds_swath['hour_ships'].values[case, 0])

utc_string = f"{year:04d}-{month:02d}-{day:02d} {hour:02d}Z"

# -------------------------------------------------
# Coordinates (km storm-relative)
# -------------------------------------------------
east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# -------------------------------------------------
# Storm center lat/lon
# -------------------------------------------------
storm_lat = ds_swath['lat_ships'].values[case, 0]
storm_lon = ds_swath['lon_ships'].values[case, 0]

# Convert positive-west longitude to standard negative
storm_lon = -abs(storm_lon)

# -------------------------------------------------
# Convert km grid to lat/lon grid
# -------------------------------------------------
deg_per_km_lat = 1 / 111.0
deg_per_km_lon = 1 / (111.0 * np.cos(np.deg2rad(storm_lat)))

lon_grid = storm_lon + east * deg_per_km_lon
lat_grid = storm_lat + north * deg_per_km_lat

lon2d, lat2d = np.meshgrid(lon_grid, lat_grid)

# -------------------------------------------------
# 1) Get shear magnitude + direction
# -------------------------------------------------
shear_mag = ds_swath['shdc_ships'].values[case, 0]
shear_dir_deg = ds_swath['sddc_ships'].values[case, 0]

theta = np.deg2rad(270 - shear_dir_deg)
ux = np.cos(theta)
uy = np.sin(theta)

# -------------------------------------------------
# 2) Plan view at 1 km
# -------------------------------------------------
height_idx = np.argmin(np.abs(height - 1.0))

plan_view = ds_swath['recentered_reflectivity'].values[
    case, :, :, height_idx
]

# -------------------------------------------------
# 3) Build shear-oriented cross section
# -------------------------------------------------
max_radius = 200  # km
n_points = 400

s = np.linspace(-max_radius, max_radius, n_points)

x_line = s * ux
y_line = s * uy

interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_section = np.zeros((len(height), n_points))

for k in range(len(height)):
    slice2d = ds_swath['recentered_reflectivity'].values[
        case, :, :, k
    ]
    
    interp.values = slice2d
    points = np.column_stack((y_line, x_line))
    cross_section[k, :] = interp(points)

# -------------------------------------------------
# 4) Plot
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# ---- LEFT: Plan View (Geographic) ----
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[lon_grid[0], lon_grid[-1], lat_grid[0], lat_grid[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

# Shear line (convert to lat/lon)
shear_lon = storm_lon + x_line * deg_per_km_lon
shear_lat = storm_lat + y_line * deg_per_km_lat

axes[0].plot(shear_lon, shear_lat, color='black', linewidth=2)

arrow_scale = 50
axes[0].arrow(
    storm_lon,
    storm_lat,
    ux * arrow_scale * deg_per_km_lon,
    uy * arrow_scale * deg_per_km_lat,
    color='white',
    width=0.05
)

# Add subtle lat/lon dots
axes[0].scatter(
    lon2d[::10, ::10],
    lat2d[::10, ::10],
    s=5,
    color='white',
    alpha=0.6
)

axes[0].set_title('Plan View (1 km)')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
fig.colorbar(im1, ax=axes[0], label='dBZ')

# ---- RIGHT: Shear-Aligned Cross Section ----
im2 = axes[1].pcolormesh(
    s,
    height,
    cross_section,
    shading='auto',
    cmap='turbo',
    vmin=0, vmax=60
)

axes[1].set_title('Vertical Cross Section (Along Shear)')
axes[1].set_xlabel('Distance Along Shear (km)')
axes[1].set_ylabel('Height (km)')
axes[1].grid(color='white', alpha=0.3, linestyle='--')

fig.colorbar(im2, ax=axes[1], label='dBZ')

# ---- Overall Title ----
fig.suptitle(
    f"Shear-Oriented Cross Section | {utc_string} UTC",
    fontsize=14
)



plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# now with a non-shear oriented cross section

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator

case = 358

# =================================================
# USER-DEFINED CROSS SECTION DIRECTION (0–360 FROM)
# =================================================
cross_dir_deg = 145   # <-- CHANGE THIS (meteorological direction FROM)

# -------------------------------------------------
# Get UTC time
# -------------------------------------------------
year = int(ds_swath['year_ships'].values[case, 0])
month = int(ds_swath['month_ships'].values[case, 0])
day = int(ds_swath['day_ships'].values[case, 0])
hour = int(ds_swath['hour_ships'].values[case, 0])

utc_string = f"{year:04d}-{month:02d}-{day:02d} {hour:02d}Z"

# -------------------------------------------------
# Coordinates (km storm-relative)
# -------------------------------------------------
east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# -------------------------------------------------
# Storm center lat/lon
# -------------------------------------------------
storm_lat = ds_swath['lat_ships'].values[case, 0]
storm_lon = ds_swath['lon_ships'].values[case, 0]
storm_lon = -abs(storm_lon)  # positive-west → negative

# -------------------------------------------------
# Convert km grid to lat/lon
# -------------------------------------------------
deg_per_km_lat = 1 / 111.0
deg_per_km_lon = 1 / (111.0 * np.cos(np.deg2rad(storm_lat)))

lon_grid = storm_lon + east * deg_per_km_lon
lat_grid = storm_lat + north * deg_per_km_lat
lon2d, lat2d = np.meshgrid(lon_grid, lat_grid)

# -------------------------------------------------
# Convert meteorological direction (FROM)
# to math angle (TOWARD, CCW from +x axis)
# -------------------------------------------------
theta = np.deg2rad(270 - cross_dir_deg)

ux = np.cos(theta)
uy = np.sin(theta)

# -------------------------------------------------
# Plan view at 1 km
# -------------------------------------------------
height_idx = np.argmin(np.abs(height - 1.0))

plan_view = ds_swath['recentered_reflectivity'].values[
    case, :, :, height_idx
]

# -------------------------------------------------
# Build cross section in chosen direction
# -------------------------------------------------
max_radius = 200
n_points = 400

s = np.linspace(-max_radius, max_radius, n_points)

x_line = s * ux
y_line = s * uy

interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_section = np.zeros((len(height), n_points))

for k in range(len(height)):
    slice2d = ds_swath['recentered_reflectivity'].values[
        case, :, :, k
    ]
    
    interp.values = slice2d
    points = np.column_stack((y_line, x_line))
    cross_section[k, :] = interp(points)

# -------------------------------------------------
# Plot
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# ---- Plan View ----
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[lon_grid[0], lon_grid[-1], lat_grid[0], lat_grid[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

# Convert cross line to lat/lon
line_lon = storm_lon + x_line * deg_per_km_lon
line_lat = storm_lat + y_line * deg_per_km_lat

axes[0].plot(line_lon, line_lat, color='black', linewidth=2)

axes[0].arrow(
    storm_lon,
    storm_lat,
    ux * 50 * deg_per_km_lon,
    uy * 50 * deg_per_km_lat,
    color='white',
    width=0.05
)

# Lat/lon dots
axes[0].scatter(
    lon2d[::10, ::10],
    lat2d[::10, ::10],
    s=5,
    color='white',
    alpha=0.6
)

axes[0].set_title('Plan View (1 km)')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
fig.colorbar(im1, ax=axes[0], label='dBZ')

# ---- Cross Section ----
im2 = axes[1].pcolormesh(
    s,
    height,
    cross_section,
    shading='auto',
    cmap='turbo',
    vmin=0, vmax=60
)

axes[1].set_title(f'Cross Section (FROM {cross_dir_deg}°)')
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('Height (km)')
axes[1].grid(color='white', alpha=0.3, linestyle='--')

fig.colorbar(im2, ax=axes[1], label='dBZ')

# ---- Overall Title ----
fig.suptitle(
    f"User-Oriented Cross Section | {utc_string} UTC",
    fontsize=14
)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import glob
import pandas as pd

# --- Load all WWLLN CSV files ---
lightning_files = sorted(
    glob.glob("/nfs/home11/ugrad/2024/es348865/honors_research/250523/WWLLN_201409*.csv")
)

# Concatenate into one DataFrame
df_light_all = pd.concat([pd.read_csv(f) for f in lightning_files])

# Fix longitude to -180 → 180
df_light_all["longitude"] = ((df_light_all["longitude"] + 180) % 360) - 180

# Make datetime column
df_light_all["datetime"] = pd.to_datetime(df_light_all["datetime"], format="%Y/%m/%d %H:%M:%S.%f")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

case = 351


# =================================================
# USER SETTINGS
# =================================================
cross_dir_deg = 100      # Cross section direction (meteorological FROM)
lightning_window_min = 15
max_radius = 200

# -------------------------------------------------
# Get UTC time
# -------------------------------------------------
year = int(ds_swath['year_ships'].values[case, 0])
month = int(ds_swath['month_ships'].values[case, 0])
day = int(ds_swath['day_ships'].values[case, 0])
hour = int(ds_swath['hour_ships'].values[case, 0])

storm_time = pd.Timestamp(year, month, day, hour)
utc_string = storm_time.strftime("%Y-%m-%d %HZ")

# -------------------------------------------------
# Storm center lat/lon
# -------------------------------------------------
storm_lat = ds_swath['lat_ships'].values[case, 0]
storm_lon = -abs(ds_swath['lon_ships'].values[case, 0])

# -------------------------------------------------
# Coordinates (storm-relative km grid)
# -------------------------------------------------
east = ds_swath['eastward_distance'].values
north = ds_swath['northward_distance'].values
height = ds_swath['height'].values

# -------------------------------------------------
# Convert km grid to lat/lon
# -------------------------------------------------
deg_per_km_lat = 1 / 111.0
deg_per_km_lon = 1 / (111.0 * np.cos(np.deg2rad(storm_lat)))

lon_grid = storm_lon + east * deg_per_km_lon
lat_grid = storm_lat + north * deg_per_km_lat
lon2d, lat2d = np.meshgrid(lon_grid, lat_grid)

# -------------------------------------------------
# User-defined cross section direction
# -------------------------------------------------
theta = np.deg2rad(270 - cross_dir_deg)
ux = np.cos(theta)
uy = np.sin(theta)

# -------------------------------------------------
# Plan view at 1 km
# -------------------------------------------------
height_idx = np.argmin(np.abs(height - 1.0))
plan_view = ds_swath['recentered_reflectivity'].values[
    case, :, :, height_idx
]

# -------------------------------------------------
# Build cross section
# -------------------------------------------------
n_points = 400
s = np.linspace(-max_radius, max_radius, n_points)
x_line = s * ux
y_line = s * uy

interp = RegularGridInterpolator(
    (north, east),
    np.zeros((len(north), len(east))),
    bounds_error=False,
    fill_value=np.nan
)

cross_section = np.zeros((len(height), n_points))

for k in range(len(height)):
    slice2d = ds_swath['recentered_reflectivity'].values[
        case, :, :, k
    ]
    interp.values = slice2d
    points = np.column_stack((y_line, x_line))
    cross_section[k, :] = interp(points)

# -------------------------------------------------
# 🔥 WWLLN LIGHTNING FILTERING
# -------------------------------------------------
start = storm_time - pd.Timedelta(minutes=lightning_window_min)
end = storm_time + pd.Timedelta(minutes=lightning_window_min)

df_light_sub = df_light_all[
    (df_light_all["datetime"] >= start) &
    (df_light_all["datetime"] <= end)
].copy()

# Convert lightning to storm-relative km
df_light_sub["x_km"] = (df_light_sub["longitude"] - storm_lon) / deg_per_km_lon
df_light_sub["y_km"] = (df_light_sub["latitude"] - storm_lat) / deg_per_km_lat

# Keep within plotting radius
df_light_sub = df_light_sub[
    (df_light_sub["x_km"]**2 + df_light_sub["y_km"]**2) <= max_radius**2
]

print(f"Lightning strikes in ±{lightning_window_min} min: {len(df_light_sub)}")

# -------------------------------------------------
# Plot
# -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# ---- PLAN VIEW ----
im1 = axes[0].imshow(
    plan_view,
    origin='lower',
    extent=[east[0], east[-1], north[0], north[-1]],
    cmap='turbo',
    vmin=0, vmax=60
)

# Cross section line
axes[0].plot(x_line, y_line, color='black', linewidth=2)

# Lightning overlay
axes[0].scatter(
    df_light_sub["x_km"],
    df_light_sub["y_km"],
    s=10,
    color="magenta",
    alpha=0.7,
    label="WWLLN"
)

axes[0].set_title("Plan View (1 km)")
axes[0].set_xlabel("Eastward Distance (km)")
axes[0].set_ylabel("Northward Distance (km)")
axes[0].legend()

fig.colorbar(im1, ax=axes[0], label="dBZ")

# ---- CROSS SECTION ----
im2 = axes[1].pcolormesh(
    s,
    height,
    cross_section,
    shading="auto",
    cmap="turbo",
    vmin=0, vmax=60
)

axes[1].set_title(f"Cross Section (FROM {cross_dir_deg}°)")
axes[1].set_xlabel("Distance (km)")
axes[1].set_ylabel("Height (km)")
axes[1].grid(color="white", alpha=0.3, linestyle="--")

fig.colorbar(im2, ax=axes[1], label="dBZ")

# ---- Titles ----
fig.suptitle(
    f"Reflectivity + WWLLN Lightning | {utc_string}",
    fontsize=14
)

plt.tight_layout()
plt.show()